In [11]:
import os
import yaml
import logging
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.sql.functions import col, udf
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassificationModel

from utils.spark_session import get_spark_session

from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

spark = get_spark_session("08-evaluate-pipeline-artifacts")

In [12]:
test_path = os.path.join("..", "data", "train_test", "test.parquet")
config_path = os.path.join("..", "src", "config", "feature_config.yaml")
features_selected_path = os.path.join("..", "src", "features", "selected", "features_selected.yaml")
model_path = os.path.join("..", "models", "final_gbt_model")

In [13]:
df = spark.read.parquet(test_path)
with open(config_path, 'r') as f:
    feature_config = yaml.safe_load(f)
with open(features_selected_path, 'r') as f:
    selected_yaml = yaml.safe_load(f)


In [14]:
target_col = [k for k, v in feature_config.items() if isinstance(v, dict) and v.get("target")][0]
selected_features = selected_yaml.get("support_random_forest", [])
available_cols = [f.name for f in df.schema.fields]
feature_cols = [f for f in selected_features if f in available_cols]

In [15]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_test = assembler.transform(df.select(*(feature_cols + [target_col]))).select("features", col(target_col).alias("label"))

df_test.show()

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[3.13043478260869...|    0|
|[1.36956521739130...|    0|
|[8.54347826086956...|    0|
|[1.82903225806451...|    0|
|[1.12580645161290...|    0|
|[0.006,0.006,186....|    0|
|[1.45806451612903...|    0|
|[3.00000000000000...|    0|
|[2.09032258064516...|    0|
|[2.76774193548387...|    0|
|[2.39354838709677...|    0|
|[1.92682926829268...|    0|
|[1.04878048780487...|    0|
|[3.43902439024390...|    0|
|[2.69268292682926...|    1|
|[2.26829268292682...|    0|
|[3.53658536585365...|    0|
|[3.65853658536585...|    0|
|[4.53658536585365...|    0|
|[4.63414634146341...|    0|
+--------------------+-----+
only showing top 20 rows



In [16]:
model = GBTClassificationModel.load(model_path)
predictions = model.transform(df_test)

In [17]:
predictions

DataFrame[features: vector, label: int, rawPrediction: vector, probability: vector, prediction: double]

In [18]:
extract_prob_1 = udf(lambda v: float(v[1]), FloatType())
predictions = predictions.withColumn("prob_1", extract_prob_1(col("probability")))

In [19]:
predictions

DataFrame[features: vector, label: int, rawPrediction: vector, probability: vector, prediction: double, prob_1: float]

In [20]:
pred_pd = predictions.select("prob_1", "prediction", "label").toPandas()
probs = pred_pd["prob_1"].values
true_labels = pred_pd["label"].values

Py4JJavaError: An error occurred while calling o410.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 21.0 failed 1 times, most recent failure: Lost task 6.0 in stage 21.0 (TID 35) (host.docker.internal executor driver): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`ProbabilisticClassificationModel$$Lambda$3408/819328225`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:198)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.h

In [ ]:
importances = model.featureImportances.toArray()
final_features = selected_features[:len(importances)]

assembler = VectorAssembler(inputCols=final_features, outputCol="features")
df_test = assembler.transform(df.select(*(final_features + [target_col])))
df_test = df_test.select("features", col(target_col).alias("label"))

engineered_features = [
    'real_amount_per_limit',
    'n_transactions_so_far',
    'amount_cumulative',
    'n_conversions_so_far',
    'pct_current_vs_total_session',
    'amount_pct_limit',
    'limit_factor_vs_tx'
]

feature_importance_df = pd.DataFrame({
    'feature': final_features,
    'importance': importances
})

feature_importance_df = feature_importance_df[feature_importance_df['importance'] > 0]

feature_importance_df['color'] = feature_importance_df['feature'].apply(
    lambda x: '#ff4c4c' if x in engineered_features else '#800000'
)

feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='importance',
    y='feature',
    data=feature_importance_df,
    palette=feature_importance_df['color'].tolist()
)
plt.title('Feature Importance - GBT Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


ValueError: All arrays must be of the same length

In [ ]:
engineered_total = feature_importance_df[
    feature_importance_df['feature'].isin(engineered_features)
]['importance'].sum()

overall_total = feature_importance_df['importance'].sum()

engineered_percentage = engineered_total / overall_total * 100
print(f"Engineered features account for {engineered_percentage:.2f}% of total importance.")

In [ ]:
pred_labels = (pred_pd["prob_1"] >= 0.5).astype(int)
report = classification_report(true_labels, pred_labels, output_dict=False)
print(report)

In [ ]:
pred_pd['score'] = (pred_pd['prob_1'] * 1000).astype(int)

pred_pd['rank_score'] = pred_pd['score'].rank(method='first')

pred_pd['decil'] = pd.qcut(pred_pd['rank_score'], q=10, labels=False)

results = (
    pred_pd.groupby('decil').label.sum() / pred_pd.label.sum()
).reset_index()

results['decil'] += 1

results['offers'] = pred_pd[pred_pd.label == 1].groupby('decil').size()
results['total_items'] = pred_pd.groupby('decil').size().astype(int)
results['no_offers'] = results['total_items'] - results['offers']
results['score (>=)'] = round(pred_pd.groupby('decil').score.min(), 3)

results.fillna(0, inplace=True)

results.sort_values(by='decil', ascending=False, inplace=True)

results = results[['decil', 'score (>=)', 'offers', 'no_offers', 'total_items', 'label']]
results = results.rename(columns={'label': 'recall'})

results['recall'] = results['recall'].cumsum()
results['precision'] = results['offers'] / results['total_items']

results['no_offers'] = results['no_offers'].cumsum()
results['total_items'] = results['total_items'].cumsum()

results.reset_index(drop=True, inplace=True)
results

Esta tabela mostra a performance do modelo ao segmentar clientes em 10 decis com base em um score preditivo, onde **Decil 10 representa os clientes com maior score (maior probabilidade de conversão)**.

* **Decis superiores** concentram os clientes com maior chance de conversão, apresentando **maior precisão**. No entanto, o **recall acumulado** ainda é relativamente baixo devido ao menor volume de clientes nesses grupos.

* **Decis inferiores** abrangem a maior parte da base de clientes e garantem **alto recall acumulado**, mas com **baixa precisão**, indicando que, embora o modelo consiga capturar praticamente todos os casos positivos, isso ocorre com muito ruído nessas faixas.

Dessa forma, **ações de marketing devem priorizar os decis superiores**, onde as taxas de conversão são mais favoráveis.


In [ ]:
spark.stop()